<a href="https://colab.research.google.com/github/aykahsay/paris-ozone-time-series-forecasting/blob/main/nimes_ozone_time_series_forecasting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Time Series Analysis and Forecasting of Ground-Level Ozone (O₃) Concentrations in Nîmes, France Using Machine Learning**

## 1. Data Collection (OpenAQ API)

The hourly ozone (O₃) measurements used in this project were collected from the [OpenAQ v3 API](https://docs.openaq.org/).

**Introduction**

Heat waves are among the most significant climate-related hazards affecting Europe, with increasing frequency and intensity due to climate change. During prolonged periods of extreme heat, high temperatures, intense solar radiation, and stagnant atmospheric conditions accelerate photochemical reactions between nitrogen oxides (NOₓ) and volatile organic compounds (VOCs), leading to elevated concentrations of ground-level ozone (O₃). Unlike stratospheric ozone, which protects the Earth from harmful ultraviolet radiation, ground-level ozone is a harmful air pollutant associated with respiratory and cardiovascular diseases, reduced agricultural productivity, and ecosystem damage.

Southern France, and the Occitanie region in particular, is repeatedly exposed to summer heat waves — events such as the exceptional June–July 2025 French heat wave (temperatures above 40°C across the south) illustrate the meteorological conditions that drive elevated ground-level ozone. Cities such as Nîmes experience prolonged heat and stagnant, sunlit conditions that favour photochemical ozone formation, tightening the link between meteorological extremes and urban air quality.

This study investigates ground-level ozone concentrations in **Nîmes, Occitanie, France**, using hourly measurements obtained from the OpenAQ monitoring station **Nîmes Gauzy (Location ID: 2162655; Sensor ID: 7774985)**. The dataset spans **2017–2019** and is used to characterise the seasonal cycle of ozone, identify the recurring warm-season peaks produced by photochemical activity, and build and compare daily forecasting models. Because meteorological drivers are not included in this OpenAQ series, the study also quantifies how much day-to-day forecast accuracy is limited by their absence — a key input to the recommendations at the end of the notebook.

The findings of this study contribute to a better understanding of how extreme heat events affect urban air quality and provide insights that can support environmental monitoring, public health protection, and climate adaptation strategies in regions increasingly exposed to severe heat waves.


In [1]:
import requests
import pandas as pd
from google.colab import userdata
API_KEY = userdata.get("OPENAQ_API_KEY")

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
headers = {
    "X-API-Key": API_KEY
}

sensor_id = 7774985      # Nîmes Gauzy O3
url = f"https://api.openaq.org/v3/sensors/{sensor_id}/measurements"

all_results = []
page = 1

while True:

    params = {
        "limit": 1000,
        "page": page
    }

    response = requests.get(url, headers=headers, params=params)
    response.raise_for_status()

    data = response.json()

    results = data["results"]

    if len(results) == 0:
        print("\nFinished downloading.")
        break

    all_results.extend(results)

    newest = results[0]["period"]["datetimeFrom"]["utc"]
    oldest = results[-1]["period"]["datetimeFrom"]["utc"]

    print(
        f"Page {page:<3} "
        f"| Records: {len(results):<4} "
        f"| {newest}  -->  {oldest}"
    )

    page += 1


In [ ]:
# Convert to DataFrame
df = pd.DataFrame(all_results)

# Extract datetime for easier analysis
df["datetime"] = df["period"].apply(
    lambda x: x["datetimeFrom"]["utc"]
)

df["datetime"] = pd.to_datetime(df["datetime"])

# Sort newest → oldest
df = df.sort_values("datetime", ascending=False).reset_index(drop=True)

print(f"\nTotal records downloaded: {len(df):,}")

df.head()

In [ ]:
df["Date"] = df["period"].apply(lambda x: x["datetimeFrom"]["utc"])
df["Date"] = pd.to_datetime(df["Date"])
df.head()

In [ ]:
ozone_df = df[["Date", "value"]].copy()

ozone_df.rename(columns={"value": "Ozone"}, inplace=True)

ozone_df.head()

In [ ]:
from google.colab import files

ozone_df.to_csv("LesHautsdeNîmes_ozone_df.csv", index=False)
files.download("LesHautsdeNîmes_ozone_df.csv")

## 2. Load Data

From here on the analysis is fully reproducible without API access — we load the previously collected data directly from `LesHautsdeNîmes_ozone_df.csv`.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.statespace.sarimax import SARIMAX

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

import seaborn as sns
import plotly.graph_objects as go

plt.rcParams["figure.figsize"] = (12, 4.5)

ozone_df = pd.read_csv("LesHautsdeNîmes_ozone_df.csv", parse_dates=["Date"])
ozone_df = ozone_df.sort_values("Date").drop_duplicates(subset="Date").set_index("Date")

print(f"Hourly observations: {len(ozone_df):,}")
print(f"Date range: {ozone_df.index.min()} -> {ozone_df.index.max()}")
ozone_df.head()

## 2.1 Dataset Metadata

Complete metadata for the mined dataset (Assignment Part a). Counts and the date range are computed directly from the loaded series so the table always reflects the data actually in memory.

In [ ]:
# --- Dataset metadata (Assignment Part a) ---
metadata = {
    "Dataset":                 "OpenAQ ground-level ozone (O₃)",
    "Location":                "Nîmes Gauzy, Nîmes, Occitanie, France",
    "Location ID":             2162655,
    "Sensor ID":               7774985,
    "Pollutant":               "Ground-level ozone (O₃)",
    "Unit":                    "µg/m³",
    "Raw temporal resolution": "Hourly (irregularly spaced)",
    "Modelling resolution":    "Daily mean (after resampling)",
    "Time period":             f"{ozone_df.index.min().date()} → {ozone_df.index.max().date()}",
    "Hourly observations":     f"{len(ozone_df):,}",
    "Source":                  "OpenAQ API v3 (https://docs.openaq.org/)",
    "Access":                  "Authenticated REST API, paginated hourly measurements",
}
pd.DataFrame(metadata.items(), columns=["Property", "Value"]).set_index("Property")

## 2.2 Data Structures & Consistency Checks

The various data structures of the mined dataset (Assignment Part b), followed by hourly-level consistency checks. `Ozone` is a floating-point measurement, indexed by a timezone-aware `DatetimeIndex`.

In [ ]:
# --- Data structures (Assignment Part b) ---
print("shape           :", ozone_df.shape)
print("index type      :", type(ozone_df.index).__name__)
print("timezone        :", ozone_df.index.tz)
print("\ndtypes:")
print(ozone_df.dtypes)
print("\ninfo:")
ozone_df.info()

# --- Hourly consistency / quality checks ---
checks = {
    "Duplicate timestamps"       : int(ozone_df.index.duplicated().sum()),
    "Negative ozone values"      : int((ozone_df["Ozone"] < 0).sum()),
    "Implausible (> 500 µg/m³)"   : int((ozone_df["Ozone"] > 500).sum()),
    "Exact zeros"                : int((ozone_df["Ozone"] == 0).sum()),
}
print("\nConsistency checks (hourly):")
display(pd.Series(checks, name="count").to_frame())

print("\nhead():");     display(ozone_df.head())
print("describe():");    display(ozone_df.describe())

## 3. Exploratory Data Analysis

In [ ]:
print(ozone_df["Ozone"].describe())

fig, ax = plt.subplots()
ozone_df["Ozone"].plot(ax=ax, lw=0.3, color="tab:blue")
ax.set_title("Raw Hourly Ozone Measurements — Nîmes")
ax.set_ylabel("Ozone (µg/m³)")
plt.tight_layout()
plt.show()

Ozone concentrations range from 0 to roughly 195 µg/m³, with a mean around 49 µg/m³. The raw sensor readings are also **irregularly spaced** — measurements are not recorded on a fixed schedule — so before any time-series modeling we resample onto a regular daily grid by averaging same-day observations.

## 4. Handling Missing Data & Resampling

We resample to a daily mean and inspect how much of the resulting calendar is missing.

In [ ]:
daily_raw = ozone_df["Ozone"].resample("D").mean().asfreq("D")

# 2016 contributes only 11 partial days at the very start of the record; drop it and start
# the series from the first full calendar year.
daily_raw = daily_raw.loc["2017-01-01":]

print(f"Daily calendar days: {len(daily_raw)}")
print(f"Missing days: {daily_raw.isna().sum()} ({daily_raw.isna().mean():.1%})")

is_null = daily_raw.isna()
gap_id = (is_null != is_null.shift()).cumsum()
gap_lengths = daily_raw[is_null].groupby(gap_id[is_null]).size().sort_values(ascending=False)
print("\nLongest gaps (consecutive missing days):")
print(gap_lengths.head(10))

Most gaps are only 1–2 days long and can safely be bridged with interpolation. A handful of much longer gaps (up to 75 days) reflect real sensor outages. We time-interpolate all gaps so the series is complete for decomposition and modeling, but we explicitly **flag days that fall inside long gaps (>14 days)** — values there are synthetic (linearly interpolated across the outage) and any patterns landing on them should be discounted rather than read as real signal.

In [ ]:
long_gap_mask = pd.Series(False, index=daily_raw.index)
gap_id2 = (daily_raw.isna() != daily_raw.isna().shift()).cumsum()
for _, grp in daily_raw[daily_raw.isna()].groupby(gap_id2[daily_raw.isna()]):
    if len(grp) > 14:
        long_gap_mask.loc[grp.index] = True

daily = daily_raw.interpolate(method="time").ffill().bfill()
print(f"Remaining missing after interpolation: {daily.isna().sum()}")
print(f"Days inside long (>14d) interpolated gaps: {long_gap_mask.sum()} ({long_gap_mask.mean():.1%})")

fig, ax = plt.subplots()
daily.plot(ax=ax, color="tab:blue", lw=0.8)
for _, grp in daily[long_gap_mask].groupby((~long_gap_mask).cumsum()[long_gap_mask]):
    ax.axvspan(grp.index.min(), grp.index.max(), color="tab:red", alpha=0.15)
ax.set_title("Daily Mean Ozone Concentration — Nîmes (2017–2019)")
ax.set_ylabel("Ozone (µg/m³)")
ax.legend(handles=[Patch(color="tab:red", alpha=0.15, label="Long interpolated gap (>14d)")])
plt.tight_layout()
plt.show()

## 4.1 Missing-Value Map & Outlier Assessment

Before modelling we visualise where the missing days fall and inspect extreme values. Ozone spikes during warm, sunny, stagnant conditions are **genuine environmental episodes**, not measurement errors, so outliers are *retained* rather than trimmed — removing them would erase exactly the heat-driven peaks this study is about.

In [ ]:
# --- Missing-value map over the daily calendar ---
fig, ax = plt.subplots(figsize=(12, 1.6))
ax.plot(daily_raw.index, daily_raw.isna().astype(int), color="tab:red", lw=0.6)
ax.set_title("Missing daily values (1 = missing)")
ax.set_yticks([0, 1]); ax.set_yticklabels(["present", "missing"])
plt.tight_layout(); plt.show()

# --- Outlier assessment on the gap-filled daily series (IQR fence) ---
q1, q3 = daily.quantile([0.25, 0.75]); iqr = q3 - q1
upper = q3 + 1.5 * iqr
n_high = int((daily > upper).sum())
print(f"IQR upper fence: {upper:.1f} µg/m³ | days above fence: {n_high} "
      f"({n_high/len(daily):.1%}) — retained as genuine ozone episodes.")

fig, ax = plt.subplots(1, 2, figsize=(12, 3.4))
ax[0].boxplot(daily.dropna(), vert=False)
ax[0].set_title("Daily ozone — boxplot"); ax[0].set_xlabel("Ozone (µg/m³)")
daily.plot.hist(bins=40, ax=ax[1], color="tab:blue", alpha=0.85)
ax[1].set_title("Daily ozone — distribution"); ax[1].set_xlabel("Ozone (µg/m³)")
plt.tight_layout(); plt.show()

## 4.2 Additional Exploratory Plots — Scatter · Heatmap · Pair · Candlestick

Beyond the raw line plot in Section 3, we examine the series' autocorrelation structure (scatter), the correlation between the target and engineered predictors (heatmap), their joint distributions (pair plot), and a weekly OHLC candlestick view. A small feature frame is built locally here so these plots are self-contained.

In [ ]:
# Local engineered features for EDA (target + lags + rolling stats)
eda_feat = pd.DataFrame({"O3": daily})
eda_feat["lag1"]        = daily.shift(1)
eda_feat["lag7"]        = daily.shift(7)
eda_feat["roll_mean_7"] = daily.shift(1).rolling(7).mean()
eda_feat["roll_std_7"]  = daily.shift(1).rolling(7).std()
eda_feat = eda_feat.dropna()

# 1) SCATTER — today vs yesterday (lag-1 persistence)
fig, ax = plt.subplots(figsize=(5.5, 5.5))
sns.scatterplot(data=eda_feat, x="lag1", y="O3", s=12, alpha=0.4, ax=ax)
ax.set_title("Lag-1 scatter: O₃(t) vs O₃(t−1)")
ax.set_xlabel("O₃ yesterday (µg/m³)"); ax.set_ylabel("O₃ today (µg/m³)")
plt.tight_layout(); plt.show()

# 2) HEATMAP — correlation of target with engineered features
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(eda_feat.corr(), cmap="coolwarm", center=0, annot=True, fmt=".2f",
            square=True, cbar_kws={"shrink": 0.7}, ax=ax)
ax.set_title("Correlation heatmap (target + engineered features)")
plt.tight_layout(); plt.show()

# 3) PAIR PLOT — joint distributions of key predictors
g = sns.pairplot(eda_feat[["O3", "lag1", "lag7", "roll_mean_7"]],
                 plot_kws={"s": 8, "alpha": 0.3}, diag_kind="kde")
g.figure.suptitle("Pair plot — target vs key predictors", y=1.02)
plt.show()

In [ ]:
# 4) CANDLESTICK — weekly OHLC constructed from the daily series
ohlc = daily.resample("W").agg(["first", "max", "min", "last"])
ohlc.columns = ["Open", "High", "Low", "Close"]

fig = go.Figure(go.Candlestick(
    x=ohlc.index, open=ohlc["Open"], high=ohlc["High"],
    low=ohlc["Low"], close=ohlc["Close"], name="Weekly O₃"))
fig.update_layout(
    title="Weekly Ozone OHLC Candlestick — Nîmes",
    yaxis_title="Ozone (µg/m³)", xaxis_rangeslider_visible=False,
    height=450, template="plotly_white")
fig.show()

## 5. Seasonal-Trend Decomposition (STL)

We decompose the gap-filled daily series into trend, seasonal (annual, period = 365 days), and residual components using STL (Seasonal-Trend decomposition using LOESS).

In [ ]:
stl_result = STL(daily, period=365, robust=True).fit()

fig = stl_result.plot()
fig.set_size_inches(12, 8)
plt.tight_layout()
plt.show()

The **trend** component shows a mild, roughly linear increase over the ~2.7-year observation window. The **seasonal** component confirms the expected annual cycle for ground-level ozone: concentrations peak in the warmer months (spring/summer), when stronger sunlight drives photochemical production, and dip in winter. The **residual** component still carries substantial day-to-day variability (largely weather-driven — temperature, wind, precipitation — none of which are in this dataset), which is the main limiting factor for daily-level forecast accuracy explored later in this notebook.

## 6. Stationarity Testing

We test both the level series and its first difference with the Augmented Dickey-Fuller (ADF) test (null hypothesis: unit root / non-stationary) and the KPSS test (null hypothesis: stationary), since the two tests have complementary blind spots.

In [ ]:
def stationarity_report(series, label):
    adf_stat, adf_p, *_ = adfuller(series.dropna(), autolag="AIC")
    kpss_stat, kpss_p, *_ = kpss(series.dropna(), regression="c", nlags="auto")
    print(f"{label:>10s} | ADF stat={adf_stat:7.3f}  p={adf_p:.4f}  | KPSS stat={kpss_stat:6.3f}  p={kpss_p:.4f}")

stationarity_report(daily, "Level")
stationarity_report(daily.diff(), "1st diff")

At the level, the ADF test rejects the unit-root null (p < 0.05) and the KPSS test fails to reject its stationarity null (p > 0.05) — the two tests **agree that the series is already stationary**, consistent with STL's near-flat trend. First-differencing pushes both tests even further toward stationarity. We let AIC-based order selection (next section) decide whether differencing is still worth including in the forecasting model.

## 7. ACF / PACF Analysis

Autocorrelation and partial autocorrelation of the differenced series help identify candidate ARIMA orders.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(daily.diff().dropna(), lags=40, ax=axes[0])
plot_pacf(daily.diff().dropna(), lags=40, ax=axes[1])
plt.tight_layout()
plt.show()

Both ACF and PACF show significant spikes mainly at lags 1–2, then cut off sharply into the confidence band, suggesting a low-order ARMA structure (p, q ≤ 2) is sufficient — which we confirm with an AIC grid search below.

## 8. Train / Validation / Test Split

We split the series chronologically into three parts, in proportions 60% / 20% / 20%:

- **Train** — used to fit each model.
- **Validation** — used to select each model's hyperparameters (SARIMA order, XGBoost's number of boosting rounds) by out-of-sample forecast accuracy, never seen during fitting.
- **Test** — used exactly once, at the end, for a final unbiased evaluation. After hyperparameters are locked in on the validation set, each model is refit on train + validation combined so it can use as much history as possible before forecasting the test period.

Every model below reports its own **prediction** and **evaluation** on both the validation and test sets, so results are directly comparable model-by-model.

In [ ]:
n = len(daily)
n_test = int(n * 0.2)
n_val = int(n * 0.2)

train = daily.iloc[: n - n_val - n_test]
val = daily.iloc[n - n_val - n_test : n - n_test]
test = daily.iloc[n - n_test :]
trainval = pd.concat([train, val])

print(f"Train:      {len(train)} days, {train.index.min().date()} -> {train.index.max().date()}")
print(f"Validation: {len(val)} days, {val.index.min().date()} -> {val.index.max().date()}")
print(f"Test:       {len(test)} days, {test.index.min().date()} -> {test.index.max().date()}")

def forecast_metrics(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / np.where(y_true == 0, np.nan, y_true))) * 100
    r2 = r2_score(y_true, y_pred)
    return {"MAE": mae, "RMSE": rmse, "MAPE (%)": mape, "R²": r2}

def show_metrics(name):
    return pd.DataFrame(
        {"Validation": val_results[name], "Test": test_results[name]}
    ).T

val_results, test_results = {}, {}
val_preds, test_preds = {}, {}

## 9. Baseline Models

Two simple baselines: a **naive** forecast (repeat the last observed value) and a **seasonal naive** forecast (repeat the value from 7 days earlier). Every subsequent model needs to clearly beat these on both validation and test.

In [ ]:
def naive_forecast(history, horizon):
    return pd.Series(history.iloc[-1], index=horizon.index)

def seasonal_naive_forecast(history, horizon, season=7):
    return pd.Series(np.resize(history.iloc[-season:].values, len(horizon)), index=horizon.index)

val_preds["Naive"] = naive_forecast(train, val)
test_preds["Naive"] = naive_forecast(trainval, test)
val_preds["Seasonal Naive (7d)"] = seasonal_naive_forecast(train, val)
test_preds["Seasonal Naive (7d)"] = seasonal_naive_forecast(trainval, test)

for name in ["Naive", "Seasonal Naive (7d)"]:
    val_results[name] = forecast_metrics(val, val_preds[name])
    test_results[name] = forecast_metrics(test, test_preds[name])

pd.concat({name: show_metrics(name) for name in ["Naive", "Seasonal Naive (7d)"]})

## 10. SARIMA Model

A full seasonal ARIMA with an annual seasonal period (`s = 365`) is not computationally tractable with a state-space SARIMAX implementation. Instead we model the **weekly** seasonal pattern directly in the seasonal ARIMA term (`s = 7`) and capture the **annual** cycle with a pair of Fourier terms (sine/cosine at the annual frequency) supplied as exogenous regressors.

**Model selection:** we grid-search small (p, d, q) x (P, Q) combinations, fit each on the training set only, forecast the validation horizon, and keep the combination with the lowest validation RMSE — i.e. we select the order by genuine out-of-sample forecast accuracy rather than in-sample AIC.

In [ ]:
def fourier_terms(index, period=365.25, K=2):
    t = np.arange(len(index))
    terms = {}
    for k in range(1, K + 1):
        terms[f"sin{k}"] = np.sin(2 * np.pi * k * t / period)
        terms[f"cos{k}"] = np.cos(2 * np.pi * k * t / period)
    return pd.DataFrame(terms, index=index)

exog_full = fourier_terms(daily.index)
exog_train, exog_val, exog_test = exog_full.loc[train.index], exog_full.loc[val.index], exog_full.loc[test.index]
exog_trainval = pd.concat([exog_train, exog_val])

# maxiter is raised from the statsmodels default (50) to 200: with the default, several
# candidate orders stop before the optimizer actually converges (ConvergenceWarning), which
# would silently bias order selection towards models that merely fit fast rather than well.
best_rmse, best_order, best_seasonal = np.inf, None, None
for p in range(3):
    for d in [0, 1]:
        for q in range(3):
            for P in range(2):
                for Q in range(2):
                    try:
                        fit = SARIMAX(
                            train, order=(p, d, q), seasonal_order=(P, 0, Q, 7),
                            exog=exog_train, enforce_stationarity=False, enforce_invertibility=False,
                        ).fit(disp=False, maxiter=200)
                        pred = fit.get_forecast(steps=len(val), exog=exog_val).predicted_mean
                        rmse = np.sqrt(mean_squared_error(val.values, pred.values))
                        if rmse < best_rmse:
                            best_rmse = rmse
                            best_order, best_seasonal = (p, d, q), (P, 0, Q, 7)
                    except Exception:
                        continue

print(f"Best order: {best_order}  seasonal_order: {best_seasonal}  validation RMSE: {best_rmse:.2f}")

**Prediction (validation):** fit on the training set only, forecast the validation horizon.

In [ ]:
sarima_val_fit = SARIMAX(
    train, order=best_order, seasonal_order=best_seasonal,
    exog=exog_train, enforce_stationarity=False, enforce_invertibility=False,
).fit(disp=False, maxiter=200)

sarima_val_pred = sarima_val_fit.get_forecast(steps=len(val), exog=exog_val).predicted_mean
sarima_val_pred.index = val.index
val_preds["SARIMA"] = sarima_val_pred
val_results["SARIMA"] = forecast_metrics(val, sarima_val_pred)

fig, ax = plt.subplots()
val.plot(ax=ax, label="Actual", color="black")
sarima_val_pred.plot(ax=ax, label="SARIMA forecast", color="tab:orange")
ax.set_title("SARIMA — Validation Period")
ax.set_ylabel("Ozone (µg/m³)")
ax.legend()
plt.tight_layout()
plt.show()

**Prediction (test):** with the order fixed by validation, refit on train + validation combined and forecast the held-out test period exactly once.

In [ ]:
sarima_final_fit = SARIMAX(
    trainval, order=best_order, seasonal_order=best_seasonal,
    exog=exog_trainval, enforce_stationarity=False, enforce_invertibility=False,
).fit(disp=False, maxiter=200)

sarima_test_pred = sarima_final_fit.get_forecast(steps=len(test), exog=exog_test).predicted_mean
sarima_test_pred.index = test.index
test_preds["SARIMA"] = sarima_test_pred
test_results["SARIMA"] = forecast_metrics(test, sarima_test_pred)

fig, ax = plt.subplots()
test.plot(ax=ax, label="Actual", color="black")
sarima_test_pred.plot(ax=ax, label="SARIMA forecast", color="tab:orange")
ax.set_title("SARIMA — Test Period")
ax.set_ylabel("Ozone (µg/m³)")
ax.legend()
plt.tight_layout()
plt.show()

**Evaluation (SARIMA):**

In [ ]:
show_metrics("SARIMA")

## 11. Machine Learning Model (XGBoost)

As an alternative to the classical SARIMA approach, we train a gradient-boosted tree model on engineered features: multiple lags, rolling statistics, calendar variables (day-of-week, month), and the same Fourier terms used above for annual seasonality.

In [ ]:
def build_features(series, K=2, period=365.25):
    feat = pd.DataFrame({"y": series})
    for lag in [1, 2, 3, 7, 14, 21]:
        feat[f"lag{lag}"] = series.shift(lag)
    feat["roll_mean_7"] = series.shift(1).rolling(7).mean()
    feat["roll_std_7"] = series.shift(1).rolling(7).std()
    feat["roll_mean_14"] = series.shift(1).rolling(14).mean()
    feat["dow"] = series.index.dayofweek
    feat["month"] = series.index.month
    t = np.arange(len(series))
    for k in range(1, K + 1):
        feat[f"sin{k}"] = np.sin(2 * np.pi * k * t / period)
        feat[f"cos{k}"] = np.cos(2 * np.pi * k * t / period)
    return feat

feature_cols = [c for c in build_features(daily).columns if c != "y"]
full_feat = build_features(daily).dropna()

def split_xy(index):
    rows = full_feat.loc[full_feat.index.isin(index)]
    return rows[feature_cols], rows["y"]

X_train, y_train = split_xy(train.index)
X_val, y_val = split_xy(val.index)

**Model selection:** we train with early stopping against the validation set to choose the number of boosting rounds — analogous to how SARIMA's order was chosen on validation RMSE above.

In [ ]:
xgb_val_model = XGBRegressor(
    n_estimators=1000, max_depth=3, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, random_state=42,
    early_stopping_rounds=30, eval_metric="rmse",
)
xgb_val_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

best_iteration = xgb_val_model.best_iteration
print(f"Selected number of boosting rounds: {best_iteration}")

def recursive_forecast(model, history, horizon_index):
    history = history.copy()
    preds = []
    for date in horizon_index:
        extended = pd.concat([history, pd.Series([np.nan], index=[date])])
        x_row = build_features(extended).loc[[date], feature_cols]
        pred = model.predict(x_row)[0]
        preds.append(pred)
        history.loc[date] = pred
    return pd.Series(preds, index=horizon_index)

**Prediction (validation):** forecast the validation horizon recursively, using only training data as history (each predicted value is fed back in to build the next step's lag features, so the model never sees true validation values).

In [ ]:
xgb_val_pred = recursive_forecast(xgb_val_model, train, val.index)
val_preds["XGBoost"] = xgb_val_pred
val_results["XGBoost"] = forecast_metrics(val, xgb_val_pred)

fig, ax = plt.subplots()
val.plot(ax=ax, label="Actual", color="black")
xgb_val_pred.plot(ax=ax, label="XGBoost forecast", color="tab:green")
ax.set_title("XGBoost — Validation Period")
ax.set_ylabel("Ozone (µg/m³)")
ax.legend()
plt.tight_layout()
plt.show()

**Prediction (test):** with the number of rounds fixed by validation, refit on train + validation combined and forecast the test period recursively.

In [ ]:
X_trainval, y_trainval = split_xy(trainval.index)

xgb_final_model = XGBRegressor(
    n_estimators=best_iteration, max_depth=3, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, random_state=42,
)
xgb_final_model.fit(X_trainval, y_trainval)

xgb_test_pred = recursive_forecast(xgb_final_model, trainval, test.index)
test_preds["XGBoost"] = xgb_test_pred
test_results["XGBoost"] = forecast_metrics(test, xgb_test_pred)

fig, ax = plt.subplots()
test.plot(ax=ax, label="Actual", color="black")
xgb_test_pred.plot(ax=ax, label="XGBoost forecast", color="tab:green")
ax.set_title("XGBoost — Test Period")
ax.set_ylabel("Ozone (µg/m³)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
importances = pd.Series(xgb_final_model.feature_importances_, index=feature_cols).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
importances.plot(kind="barh", ax=ax, color="tab:blue")
ax.invert_yaxis()
ax.set_title("XGBoost Feature Importance")
plt.tight_layout()
plt.show()

**Evaluation (XGBoost):**

In [ ]:
show_metrics("XGBoost")

## 12. Model Comparison & Evaluation

All four models side by side on both validation and test. Note that MAPE is unstable whenever actual ozone is near zero — several winter days in the validation period have values close to 0 µg/m³, so MAPE spikes there even for accurate forecasts in absolute terms; MAE and RMSE are the more reliable metrics for this dataset.

In [ ]:
model_order = ["Naive", "Seasonal Naive (7d)", "SARIMA", "XGBoost"]

print("Validation set")
display(pd.DataFrame(val_results).T.loc[model_order].sort_values("RMSE"))

print("\nTest set")
display(pd.DataFrame(test_results).T.loc[model_order].sort_values("RMSE"))

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
train.iloc[-30:].plot(ax=ax, label="Train (last 30d)", color="0.6")
pd.concat([val, test]).plot(ax=ax, label="Actual", color="black", lw=1.2)
pd.concat([val_preds["SARIMA"], test_preds["SARIMA"]]).plot(ax=ax, label="SARIMA forecast", color="tab:orange")
pd.concat([val_preds["XGBoost"], test_preds["XGBoost"]]).plot(ax=ax, label="XGBoost forecast", color="tab:green")
ax.axvline(val.index.min(), color="0.3", ls="--", lw=1)
ax.axvline(test.index.min(), color="0.3", ls="--", lw=1)
ax.set_title("Validation + Test Forecasts vs. Actual Ozone")
ax.set_ylabel("Ozone (µg/m³)")
ax.legend()
plt.tight_layout()
plt.show()

## 13. Discussion

The comparison tables and forecast overlay support a few clear observations:

- **Both models beat the baselines, but for different reasons.** SARIMA encodes the annual cycle explicitly (weekly seasonal term + annual Fourier regressors), so it produces a smooth seasonal trajectory that is hard to beat over a multi-week horizon. XGBoost instead learns from lag and rolling features; it can react to recent local structure but, because it forecasts recursively, small step-ahead errors compound over the horizon, which is why it typically tracks validation well yet generalises slightly less cleanly to the test period.

- **Seasonality is the dominant, learnable signal.** The STL decomposition already showed a strong, regular annual cycle and only a mild trend. That is exactly the part of the series both models capture well — and it is why even the *seasonal*-naive baseline is respectable while the plain naive baseline is not.

- **Heat-wave / episodic peaks are where every model struggles.** The sharp warm-season spikes are driven by short-lived meteorological conditions (temperature, solar radiation, stagnant air). None of these are in the dataset, so they surface as large STL residuals and as the irreducible gap between each model's error and zero. No amount of additional lag engineering closes that gap; it is a *data* limitation, not a *model* one.

- **Metric choice matters here.** MAE and RMSE are the trustworthy headline metrics. MAPE is inflated by near-zero winter values, and R² (added to the metrics) is most useful for confirming that the models explain substantially more variance than the naive baseline rather than as an absolute target.

## 14. Interactive Dashboard

An interactive dashboard satisfies the assignment's dashboard requirement in two forms:

1. **Inline (below):** a Plotly figure with a range slider that overlays the actual series against the SARIMA and XGBoost forecasts — zoom and pan directly in the notebook.
2. **Standalone Streamlit app (`ozone_dashboard.py`):** a multi-tab dashboard (EDA, seasonality, forecast horizon slider). Run it with:

   ```bash
   pip install streamlit pandas numpy plotly statsmodels
   streamlit run ozone_dashboard.py
   ```
   Then upload `LesHautsdeNîmes_ozone_df.csv` (columns `Date`, `Ozone`) via the sidebar.

In [ ]:
# Inline interactive dashboard: actual vs SARIMA vs XGBoost (val + test)
actual = pd.concat([val, test])
sarima_all = pd.concat([val_preds["SARIMA"], test_preds["SARIMA"]])
xgb_all    = pd.concat([val_preds["XGBoost"], test_preds["XGBoost"]])

fig = go.Figure()
fig.add_scatter(x=train.iloc[-30:].index, y=train.iloc[-30:].values,
                name="Train (last 30d)", line=dict(color="#9ca3af", width=1))
fig.add_scatter(x=actual.index, y=actual.values, name="Actual",
                line=dict(color="black", width=1.6))
fig.add_scatter(x=sarima_all.index, y=sarima_all.values, name="SARIMA",
                line=dict(color="#ea580c", width=1.6))
fig.add_scatter(x=xgb_all.index, y=xgb_all.values, name="XGBoost",
                line=dict(color="#16a34a", width=1.6))
fig.add_vline(x=test.index.min(), line_dash="dash", line_color="#6b7280")
fig.update_layout(
    title="Interactive Forecast Dashboard — Actual vs SARIMA vs XGBoost",
    yaxis_title="Ozone (µg/m³)", height=460, template="plotly_white",
    hovermode="x unified",
    xaxis=dict(rangeslider=dict(visible=True), type="date"))
fig.show()

## 15. Recommendations — Improving the Dataset & Methods

**Improving the dataset (highest leverage first)**

- **Add meteorological covariates.** Daily ozone is photochemically driven, so temperature, solar radiation, wind speed, relative humidity, and boundary-layer stability would address the single largest source of unexplained variance identified in the discussion. ERA5 / Copernicus reanalysis (or a co-located weather station) provides these at daily resolution and aligns cleanly to this series.
- **Add precursor pollutants.** NOₓ and VOC concentrations are the chemical inputs to ozone formation; where OpenAQ exposes co-located sensors, including them would let the model reason about formation directly rather than only through seasonality.
- **Extend and densify the record.** The series is ~2.7 years with roughly a quarter of days missing (including two multi-month outages). A longer, more complete record would both stabilise seasonal estimation and make higher-capacity models viable.
- **Incorporate nearby stations.** Pooling several Occitanie stations (spatial context) would help distinguish local sensor noise from genuine regional ozone episodes and support gap-filling that is better than linear interpolation.

**Improving the methods**

- **Turn the models into exogenous-driven forecasts.** Once weather is available, feed it as exogenous regressors to SARIMAX and as features to XGBoost; this is expected to be the biggest accuracy gain and is a small code change given the current structure.
- **Add a probabilistic / interval layer.** Report prediction intervals (SARIMA already exposes them; quantile or conformal wrappers for XGBoost) so forecasts communicate uncertainty around heat-wave peaks rather than point estimates alone.
- **Benchmark modern sequence models — but honestly.** Prophet, LSTM/GRU, and Transformer-family models (TFT, N-HiTS) are worth trying *only after the record is extended*; with ~2.7 years they are prone to overfit and unlikely to beat the current SARIMA/XGBoost, so they belong under future work rather than as immediate next steps.
- **Strengthen validation.** Replace the single chronological split with rolling-origin (expanding-window) cross-validation to obtain more robust, less split-dependent error estimates.
- **Improve gap handling.** For long outages, compare linear interpolation against seasonal or model-based imputation, and quantify sensitivity of the final metrics to the imputation choice.

## 16. Conclusion

- Both SARIMA and XGBoost clearly outperform the naive and seasonal-naive baselines on both validation and test, confirming that the trend and seasonal structure identified in the STL decomposition is genuinely learnable and useful for forecasting.
- SARIMA (weekly seasonal term + Fourier terms for the annual cycle) achieved the best test-set RMSE/MAE of the models tried. XGBoost (lag/rolling/calendar features, recursive forecasting) was competitive on validation but generalized slightly less well to the test period.
- SARIMA's forecast is a smooth curve dominated by the seasonal/trend signal; XGBoost tracks a bit more local structure, but both largely miss the sharp day-to-day swings in the actual series.
- Using a held-out validation set (rather than in-sample AIC or default hyperparameters) to choose the SARIMA order and the XGBoost boosting-round count made model selection genuinely out-of-sample, and the subsequent test-set evaluation — touched only once, after all choices were locked in — gives an honest estimate of how each model would perform on truly new data.
- The residual component from the STL decomposition, and the gap between every model's error and zero, point to the same limitation: **daily ozone is heavily driven by short-term weather** (temperature, wind, sunlight, precipitation), none of which is available as a predictor in this dataset. Incorporating weather covariates would likely be the single biggest lever for improving forecast accuracy further.
- A secondary limitation is data completeness: roughly a quarter of calendar days in 2017–2019 were missing and had to be interpolated, including two multi-month sensor outages, both of which fall entirely within the training period and do not contaminate the validation or test sets.